## Machine Learnign and EEG Classification
Dataset: PhysioNet EEG Motor Imagery (MNE-Python) + Allen Mouse Brain Atlas
Models: SVM, Random forest, K-Means, EEg Transformer (PyTorch)

## Imports
Import all required libraries. MNE-Python handles EEG signal processing, scikit-learn provides classical ML classifiers, and Pytorch (later imported) provides deep learning for the Transformer model.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import sqlite3
import warnings
warnings.filterwarnings('ignore')

mne.set_log_level('WARNING')  # suppress verbose MNE output
print("All imports successful")

## Download PhysioNet EEG data
Download PhysioNet EEG Motor Imagery dataset using MNE's built in downloader. Subject 1, runs 6 and 10 contain motor imagery trails. Data is cached locallu after first download. 64 channels, 160 Hz sampling rate, 250 sec total.

In [ ]:
from mne.datasets import eegbci

subject = 1
runs = [6, 10]

print(f"Downloading EEG data for subject {subject}, runs {runs}...")

# update_path=True tells MNE to save the path without asking
raw_files = [mne.io.read_raw_edf(f, preload=True)
             for f in eegbci.load_data(subject, runs, update_path=True)]

raw = mne.concatenate_raws(raw_files)

print(f"\nData loaded!")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f} seconds")
print(f"Channel names (first 10): {raw.ch_names[:10]}")

## Preprocess EEG Signal
Preprocess raw EEG signal: standardize electrode names, apply 8-40 Hz bandpass filter to isolate motor relevant frequencies (alpha and beta bands), and extract event markers. T0 = rest, T1 = imagines left fist, and T2 = imagined right fist.

In [ ]:
# raw EEG is noisy - we need to:
# 1. standardize channel names
# 2. apply a bandpass filter to keep only relevant frequencies
# 3. extract events (when each trial started and what type it was)

# standardize channel names to match MNE's standard montage
eegbci.standardize(raw)

# bandpass filter: keep only 8-40 Hz
# this range captures alpha (8-12Hz) and beta (12-40Hz) waves
# which are the frequencies that change during motor imagery
raw.filter(8., 40., fir_window='hamming')

# extract events from the EEG annotations
# events is an array of [sample_index, 0, event_type]
events, event_dict = mne.events_from_annotations(raw)

print(f"Event types found: {event_dict}")
print(f"Total events: {len(events)}")
print(f"\nSample events (first 5):")
for e in events[:5]:
    print(f"  sample {e[0]}, type {e[2]}")

## Extract Epochs
Extract individual trials (epochs) from the continuous EEG recording. Each epoch is a 4 second window starting at the motor imagery cue. T1 =  imagined left fist (label 0) and T2 = imagined right fist (label 1). Result: 30 trials X 64 channels X 641 time points.

In [ ]:
# an epoch = one trial of the experiment
# each trial: subject sees a cue, imagines moving left or right fist
# we extract 0-4 seconds after each cue

# in this dataset:
# T1 = imagining left fist
# T2 = imagining right fist
# we want only T1 and T2, not rest periods

# find which event IDs correspond to T1 and T2
t1_t2 = {k: v for k, v in event_dict.items() if k in ['T1', 'T2']}
print(f"Using events: {t1_t2}")

epochs = mne.Epochs(
    raw, events,
    event_id=t1_t2,
    tmin=0.0,    # start of epoch (seconds after cue)
    tmax=4.0,    # end of epoch
    baseline=None,
    preload=True
)

print(f"\nEpochs extracted!")
print(f"Number of epochs: {len(epochs)}")
print(f"Epoch shape: {epochs.get_data().shape}")
print(f"Shape means: (n_trials, n_channels, n_timepoints)")

# get labels: 0 = left fist, 1 = right fist
labels = epochs.events[:, 2] - min(epochs.events[:, 2])
print(f"\nLabels: {labels}")
print(f"Left fist (0): {sum(labels==0)} trials")
print(f"Right fist (1): {sum(labels==1)} trials")

## Extract Band Power Features
Extract band power features from EEG epoch. For each trial, compute average signal power in alpha (8-12 Hz), beta (12-30 Hz), and gamma (30-40 Hz) bands across all 64 channels. Produces a 30x192 feature matrix (30 trials X 3 bands X 64 channels) for classical ML classification.

In [ ]:
# raw EEG timeseries is too high-dimensional for simple ML
# we extract meaningful features from each epoch:
# power in different frequency bands per channel

def extract_band_power(epoch_data, sfreq, bands):
    '''
    computes average power in each frequency band for each channel
    epoch_data: shape (n_channels, n_timepoints)
    returns: feature vector of length n_channels * n_bands
    '''
    from scipy import signal as scipy_signal
    features = []

    for band_name, (fmin, fmax) in bands.items():
        # design bandpass filter for this band
        b, a = scipy_signal.butter(4, [fmin/(sfreq/2), fmax/(sfreq/2)],
                                    btype='band')
        # filter each channel and compute power
        for ch in range(epoch_data.shape[0]):
            filtered = scipy_signal.filtfilt(b, a, epoch_data[ch])
            power = np.mean(filtered**2)  # mean power
            features.append(power)

    return np.array(features)

# frequency bands relevant to motor imagery
bands = {
    'alpha': (8, 12),   # relaxed attention, suppressed during movement
    'beta':  (12, 30),  # active thinking, motor control
    'gamma': (30, 40),  # high-level processing
}

# extract features for all epochs
epoch_data = epochs.get_data()  # shape: (n_trials, n_channels, n_timepoints)
sfreq = raw.info['sfreq']

print("Extracting features from all epochs...")
X = np.array([extract_band_power(epoch_data[i], sfreq, bands)
              for i in range(len(epoch_data))])
y = labels

print(f"Feature matrix shape: {X.shape}")
print(f"Features per trial: {X.shape[1]} ({len(bands)} bands × {epoch_data.shape[1]} channels)")
print(f"Labels shape: {y.shape}")

## Tain and Evaluate Classifiers
Train SVM and Random Forest classifiers on band power features. StandardScaler normalizes features for SVM. Stratified 80/20 split preserves class balance. SVM achieves 66.67% accuracy, consistent with BCI literature showing SVMs outperform tree methods on small EEG datasets.

In [ ]:
# normalize features: important for SVM
# StandardScaler makes each feature have mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# split into train and test sets
# with only 30 trials, we use 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# --- SVM classifier ---
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc  = svm.score(X_test, y_test)

print(f"\nSVM Results:")
print(f"  Test accuracy: {svm_acc:.2%}")
print(classification_report(y_test, svm_pred,
      target_names=['left fist', 'right fist']))

# --- Random Forest classifier ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc  = rf.score(X_test, y_test)

print(f"Random Forest Results:")
print(f"  Test accuracy: {rf_acc:.2%}")
print(classification_report(y_test, rf_pred,
      target_names=['left fist', 'right fist']))

## Cross Validation
5-fold cross validation provides more reliable accuracy estimates than a single split. Each fold uses different train/test partitions, averaging results across 5 independent evaluations. SVM: 66.67% ± 10.54%, Random Forest: 53.33% ± 12.47%. Variance reflects natural trial-to-trial variability in EEG signals.

In [ ]:
# with only 30 samples, a single train/test split is unreliable
# cross-validation splits the data 5 different ways and averages results
# this gives a much more honest accuracy estimate
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

print("5-fold cross-validation (more reliable with small dataset):")
print("─" * 45)

for name, clf in [('SVM', SVC(kernel='rbf', C=1.0)),
                   ('Random Forest', RandomForestClassifier(n_estimators=100))]:
    scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')
    print(f"{name}:")
    print(f"  Fold scores: {[f'{s:.2%}' for s in scores]}")
    print(f"  Mean: {scores.mean():.2%} ± {scores.std():.2%}")
    print()

## Feature Importance
Identify which EEG channels and frequency bands are most discriminative for motor imagery classification. C3 beta band emerges as the top feature. C3 sits over left motor cortex and beta desynchronization there during right hand imagery is a well established BSI finding. This validates the model learned genuine neural patterns.

In [ ]:
# train random forest on all data to get feature importances
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_scaled, y)

importances = rf_full.feature_importances_
n_channels  = epoch_data.shape[1]
band_names  = list(bands.keys())

# reshape importances: (n_bands, n_channels)
imp_matrix = importances.reshape(len(band_names), n_channels)

# find top 5 most important channel-band combinations
flat_idx = np.argsort(importances)[::-1][:10]
print("Top 10 most important features for left vs right fist classification:")
print("─" * 55)
for idx in flat_idx:
    band_idx = idx // n_channels
    ch_idx   = idx  % n_channels
    ch_name  = epochs.ch_names[ch_idx]
    band     = band_names[band_idx]
    print(f"  {ch_name} ({band} band): importance {importances[idx]:.4f}")

## Save ML Results to DB
Save ML results to SQLite DB. Stores both test set accuracy and 5-fold cross validation scores for SVM and Random Forest. Both metrics are stored because they tell different stores. Test accuracy is one split, cross-cal is a more reliable generalization estimate. Results feed directly into the Streamlit dashboard.

In [ ]:
conn   = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS ml_results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        model TEXT NOT NULL,
        accuracy REAL NOT NULL,
        dataset TEXT NOT NULL,
        notes TEXT
    )
''')

# cross-val scores
svm_cv = cross_val_score(
    SVC(kernel='rbf', C=1.0), X_scaled, y, cv=5
).mean()
rf_cv  = cross_val_score(
    RandomForestClassifier(n_estimators=100), X_scaled, y, cv=5
).mean()

results = [
    ('SVM',           svm_acc, 'PhysioNet EEG', 'test set accuracy'),
    ('RandomForest',  rf_acc,  'PhysioNet EEG', 'test set accuracy'),
    ('SVM_crossval',  svm_cv,  'PhysioNet EEG', '5-fold cross-validation'),
    ('RF_crossval',   rf_cv,   'PhysioNet EEG', '5-fold cross-validation'),
]

for model, acc, dataset, notes in results:
    cursor.execute('''
        INSERT INTO ml_results (model, accuracy, dataset, notes)
        VALUES (?, ?, ?, ?)
    ''', (model, round(float(acc), 4), dataset, notes))

conn.commit()
cursor.execute('SELECT * FROM ml_results')
print("ML results saved to database:")
for row in cursor.fetchall():
    print(f"  {row[1]}: {row[2]:.2%} ({row[4]})")

conn.close()

## Unsupervised Clustering of Brain Regions
Can the algorithm discover functional brain regions?

## Reload Allen Data for Clustering
Define functional_groups as ground truth for cluster validation. K-Means will not see these labels during clustering. If discovered clusters match functional groups, it confirms spatial organization reflects functional organization.

In [ ]:
# K-Means clustering on the connectivity matrix
# question: can the algorithm group brain regions by
# wiring pattern without being told what they are?

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as SS
from sklearn.metrics import silhouette_score
from allensdk.core.mouse_connectivity_cache import MouseConnectivityCache
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

# reload allen data
mcc = MouseConnectivityCache(manifest_file='../allen_cache/manifest.json')
structure_tree = mcc.get_structure_tree()
experiments    = mcc.get_experiments(dataframe=True)

acronym_to_id  = structure_tree.get_id_acronym_map()
id_to_name     = structure_tree.get_name_map()

top_regions = (
    experiments['structure_abbrev']
    .value_counts()
    .head(20)
    .index.tolist()
)

functional_groups = {
    'visual':      ['VISp', 'VISl', 'VISam', 'VISpor'],
    'motor':       ['MOs', 'MOp'],
    'hippocampal': ['CA1', 'DG', 'ENTl', 'ENTm'],
    'association': ['ACAd', 'ACAv', 'RSPv', 'CP', 'MD',
                    'PAG', 'LHA', 'SSp-bfd', 'SSs', 'retina']
}

region_coords = {}
for region in top_regions:
    region_exps = experiments[experiments['structure_abbrev'] == region]
    if len(region_exps) > 0:
        region_coords[region] = [
            region_exps['injection_x'].mean(),
            region_exps['injection_y'].mean(),
            region_exps['injection_z'].mean()
        ]

valid_regions   = [r for r in top_regions if r in region_coords]
coords_array    = np.array([region_coords[r] for r in valid_regions])
dist_matrix     = cdist(coords_array, coords_array, metric='euclidean')
epsilon         = 1e-6
strength_matrix = 1 / (dist_matrix + epsilon)
np.fill_diagonal(strength_matrix, 0)
strength_matrix /= strength_matrix.max()

print(f"Allen data reloaded: {len(valid_regions)} regions")

## K-Means Clustering
Run K-Means with K = 4 on standardized 3D injection coordinates. Clustering by physical location recovers functional organization: visual, motor, and hippocampal regions group together without any functional labels provided. Silhouette score 0.41 reflects reasonable separation for biological data where functional boundaries are inherently fuzzy.

In [ ]:
from sklearn.preprocessing import StandardScaler as SS

# use raw 3D coordinates as features
# this clusters regions by physical location in the brain
# which correlates with functional organization
coords_scaled = SS().fit_transform(coords_array)

K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(coords_scaled)

print(f"K-Means clustering with K={K} (3D spatial coordinates)")
print("─" * 50)

for cluster_id in range(K):
    members = [valid_regions[i]
               for i, c in enumerate(cluster_labels) if c == cluster_id]
    print(f"\nCluster {cluster_id}: {members}")
    for group_name, group_members in functional_groups.items():
        overlap = [m for m in members if m in group_members]
        if overlap:
            print(f"  overlaps with {group_name}: {overlap}")

# measure clustering quality
from sklearn.metrics import silhouette_score
sil_score = silhouette_score(coords_scaled, cluster_labels)
print(f"\nSilhouette score: {sil_score:.4f}")
print("(closer to 1.0 = better separated clusters)")

## Visualize Clusters with PCA
Visualize K-Means clusters using PCA to reduce 3D coordinates to 2D. PC1 (43%) and PC2 (33.3%) together capture 76.3% of total variance. Color coded clusters show clear functional anatomy: motor regions (red, bottom left); hippocampal (teal, right); visual (purple, upper left).

In [ ]:
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(coords_scaled)

plt.figure(figsize=(11, 7))
colors = ['#E8593C', '#1D9E75', '#7F77DD', '#BA7517']
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

for cluster_id in range(K):
    mask = cluster_labels == cluster_id
    plt.scatter(
        coords_2d[mask, 0],
        coords_2d[mask, 1],
        c=colors[cluster_id],
        s=150,
        label=cluster_names[cluster_id],
        zorder=3,
        edgecolors='white',
        linewidth=0.5
    )
    for i, region in enumerate(valid_regions):
        if cluster_labels[i] == cluster_id:
            plt.annotate(
                region,
                (coords_2d[i, 0], coords_2d[i, 1]),
                textcoords="offset points",
                xytext=(7, 4),
                fontsize=9
            )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Brain Region Clusters by 3D Spatial Location (K-Means, K=4)')
plt.legend()
plt.tight_layout()
plt.savefig('../data/kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")

## Save Clustering Results to DB
Save K-Means cluster assignments to the DB. Algorithm label 'kmeans_k4_spatial' precisely identifies the method, parameter, and feature type used. GROUP_CONCAT query produces a human readable cluster summary matching what the Streamlit dashboard displays.

In [ ]:
conn   = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

# clear old duplicate entries first
cursor.execute('DELETE FROM clustering_results')

for i, region in enumerate(valid_regions):
    cursor.execute('''
        INSERT INTO clustering_results (region, cluster_id, algorithm)
        VALUES (?, ?, ?)
    ''', (region, int(cluster_labels[i]), 'kmeans_k4_spatial'))

conn.commit()
cursor.execute('''
    SELECT cluster_id, GROUP_CONCAT(region, ", ") as regions
    FROM clustering_results
    GROUP BY cluster_id
    ORDER BY cluster_id
''')
print("Clustering results saved (cleaned):")
for row in cursor.fetchall():
    print(f"  Cluster {row[0]}: {row[1]}")

conn.close()

## Prepare Data for Transformer
Prepare EEG data for the Transformer model. Select 7 motor cortex channels (C3, C4, Cz, FC3, FC4, CP3, CP4) to reduce dimensionality and focus on task relevant signal. Downsample time from 641 to 81 points, normalize per trial and transpose to (trials, time, channels) format expected by PyTorch Transformer.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

epoch_data = epochs.get_data()
y_labels   = labels

# correct channel names after standardization
motor_channels = ['C3', 'C4', 'Cz', 'FC3', 'FC4', 'CP3', 'CP4']
ch_names   = epochs.ch_names
motor_idx  = [ch_names.index(ch) for ch in motor_channels if ch in ch_names]

print(f"Motor channels found: {[ch_names[i] for i in motor_idx]}")

# extract motor channels only
X_motor = epoch_data[:, motor_idx, :]
print(f"Shape after channel selection: {X_motor.shape}")

# downsample time from 641 → ~80 points
X_ds   = X_motor[:, :, ::8]
print(f"Shape after downsampling: {X_ds.shape}")

# normalize each trial
X_norm = (X_ds - X_ds.mean(axis=2, keepdims=True)) / \
         (X_ds.std(axis=2, keepdims=True) + 1e-8)

# reshape for transformer: (batch, time, features)
X_transformer = X_norm.transpose(0, 2, 1)
print(f"Transformer input shape: {X_transformer.shape}")
print(f"Meaning: (n_trials, time_steps, n_channels)")

## Data Augmentation
Define EEG data augmentation to expand 30 trials to 300 by adding Gaussian noise and amplitude scaling, both mimic natural EEG variability. This cell augments all data before splitting, which causes data leakage. Kept to document the bug. Cell 17 corrects this by splitting firt then augmenting only the training set.

In [ ]:
# with 30 trials we need more data
# augmentation creates slightly modified copies of existing trials

def augment_eeg(X, y, n_augments=5):
    '''
    creates augmented copies of each trial by:
    1. adding small gaussian noise
    2. slight time shifting
    3. amplitude scaling
    '''
    X_aug = [X]
    y_aug = [y]

    for _ in range(n_augments):
        # gaussian noise (very small)
        noise = np.random.normal(0, 0.05, X.shape)
        X_noisy = X + noise

        # amplitude scaling (±10%)
        scale = np.random.uniform(0.9, 1.1, (X.shape[0], 1, 1))
        X_scaled = X_noisy * scale

        X_aug.append(X_scaled)
        y_aug.append(y)

    return np.vstack(X_aug), np.concatenate(y_aug)

X_aug, y_aug = augment_eeg(X_transformer, y_labels, n_augments=9)
print(f"After augmentation:")
print(f"  Trials: {len(X_aug)} (was {len(X_transformer)})")
print(f"  Shape: {X_aug.shape}")
print(f"  Left fist:  {sum(y_aug==0)}")
print(f"  Right fist: {sum(y_aug==1)}")

# NOTE: this augmentation was done on all data before splitting
# and is superseded by Cell 17 which correctly splits first then augments
# kept here to show the data leakage bug that was identified and corrected

## Define EEG Transformer
Define the EEG Transformer architecture: A 4 component attention model built from scratch in PyTorch. Input projection maps 7 EEG channels to 32 dimensional model space. Learnable positional encoding injects temporal order. Two Transformer encoder layers apply self attention across 81 timesteps. global average pooling aggregates temporal output before the classification head. 20,498 parameters, deliberately small to avoid overfitting 30 trials.

In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self, n_channels, seq_len, n_classes=2,
                 d_model=32, n_heads=4, n_layers=2, dropout=0.3):
        super().__init__()

        # d_model must be divisible by n_heads
        self.d_model = d_model

        # 1. input projection: maps n_channels → d_model
        #    (like an embedding layer for EEG channels)
        self.input_proj = nn.Linear(n_channels, d_model)

        # 2. positional encoding: tells the model where in time each step is
        #    (transformers have no built-in sense of sequence order)
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model))

        # 3. transformer encoder layers: the attention mechanism
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=64,
            dropout=dropout,
            batch_first=True  # input shape: (batch, seq, features)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=n_layers)

        # 4. classification head: maps transformer output → class probabilities
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, n_classes)
        )

    def forward(self, x):
        # x shape: (batch, seq_len, n_channels)

        # project input to d_model dimensions
        x = self.input_proj(x)          # (batch, seq_len, d_model)

        # add positional encoding
        x = x + self.pos_encoding       # (batch, seq_len, d_model)

        # pass through transformer (attention happens here)
        x = self.transformer(x)         # (batch, seq_len, d_model)

        # global average pooling over time dimension
        x = x.mean(dim=1)               # (batch, d_model)

        # classify
        x = self.classifier(x)          # (batch, n_classes)
        return x

# instantiate model
n_channels = X_aug.shape[2]
seq_len    = X_aug.shape[1]

model = EEGTransformer(
    n_channels=n_channels,
    seq_len=seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.3
)

# count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"EEGTransformer built!")
print(f"Input: (batch, {seq_len} timepoints, {n_channels} channels)")
print(f"Total parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

## Train Transformer with Correct Split
Train EEG Transformer with corrected data pipeline: split original 30 trials first, then augment only the training set. Test set remains original unaugmented sata. This fixes the data leakage bug in Cell 15 where augmented copies of test trials appeared in training. Smooth loss descent confirms correct training dynamics despite small dataset size.

In [ ]:
# ORDER:
# 1. split ORIGINAL data first
# 2. augment ONLY the training set
# 3. test on original unaugmented data

from sklearn.model_selection import train_test_split as tts

# split original 30 trials
X_tr_orig, X_te_orig, y_tr_orig, y_te_orig = tts(
    X_transformer, y_labels,
    test_size=0.2, random_state=42, stratify=y_labels
)

print(f"Original split: {len(X_tr_orig)} train, {len(X_te_orig)} test")

# augment ONLY training data
X_tr_aug, y_tr_aug = augment_eeg(X_tr_orig, y_tr_orig, n_augments=9)
print(f"After augmenting train set: {len(X_tr_aug)} trials")
print(f"Test set stays original: {len(X_te_orig)} trials (no augmentation)")

# convert to tensors
X_train_t = torch.FloatTensor(X_tr_aug)
y_train_t = torch.LongTensor(y_tr_aug)
X_test_t  = torch.FloatTensor(X_te_orig)
y_test_t  = torch.LongTensor(y_te_orig)

# rebuild and retrain model fresh
model2 = EEGTransformer(
    n_channels=X_transformer.shape[2],
    seq_len=X_transformer.shape[1],
    d_model=32, n_heads=4, n_layers=2, dropout=0.3
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)

n_epochs     = 60
train_losses = []
print("\nTraining with proper split...")
print("─" * 40)

for epoch in range(n_epochs):
    model2.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model2(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f}")

# evaluate on original unaugmented test set
model2.eval()
with torch.no_grad():
    test_outputs = model2(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    accuracy2    = (predicted == y_test_t).float().mean().item()

print(f"\nTransformer test accuracy (honest): {accuracy2:.2%}")
print(f"(tested on {len(X_te_orig)} original unaugmented trials)")

# plot
plt.figure(figsize=(8, 4))
plt.plot(train_losses, color='#1D9E75')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Training Loss (Corrected Split)')
plt.tight_layout()
plt.savefig('../data/transformer_loss_corrected.png', dpi=150, bbox_inches='tight')
plt.show()

## Transformer Results to DB
Save Transformer accuracy to the DM. Final ranking: SVM 66.67% > SVM crossval 66.67% > RF crossval 56.67% > RandomForest 50% > EEGTransformer 33.33% (data-limited).

In [ ]:
conn   = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

# remove the inflated 100% result and the empty-channel result
cursor.execute("DELETE FROM ml_results WHERE model = 'EEGTransformer'")

# save the honest result
cursor.execute('''
    INSERT INTO ml_results (model, accuracy, dataset, notes)
    VALUES (?, ?, ?, ?)
''', ('EEGTransformer', round(float(accuracy2), 4),
      'PhysioNet EEG',
      f'2-layer transformer, split before augmentation, {n_epochs} epochs'))

conn.commit()
cursor.execute('SELECT model, accuracy, notes FROM ml_results ORDER BY accuracy DESC')
print("Final ML results (honest):")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]:.2%} — {row[2]}")

conn.close()

## Reload with Extended Dataset
load extended dataset with 3 runs (6,10,14) to get 45 trials instead of 30. Applies identical preprocessing pipeline: standardize → 8-40 Hz filter → epoch → motor channel selection → downsample → normalize → transpose. More trials gives more reliable Transformer evaluation. 9 test trials instead of 6.

In [ ]:
# runs 6,10,14 = imagine left/right fist (3 sessions)
# this gives us ~90 trials instead of 30
runs_extended = [6, 10, 14]

print("Loading extended dataset...")
raw_files2 = [mne.io.read_raw_edf(f, preload=True)
              for f in eegbci.load_data(subject, runs_extended,
                                         update_path=True)]
raw2 = mne.concatenate_raws(raw_files2)
eegbci.standardize(raw2)
raw2.filter(8., 40., fir_window='hamming')

events2, event_dict2 = mne.events_from_annotations(raw2)
t1_t2_2 = {k: v for k, v in event_dict2.items() if k in ['T1', 'T2']}

epochs2 = mne.Epochs(raw2, events2, event_id=t1_t2_2,
                     tmin=0.0, tmax=4.0, baseline=None, preload=True)

labels2 = epochs2.events[:, 2] - min(epochs2.events[:, 2])

print(f"Epochs: {len(epochs2)}")
print(f"Left fist:  {sum(labels2==0)}")
print(f"Right fist: {sum(labels2==1)}")

# rebuild feature matrix
epoch_data2  = epochs2.get_data()
motor_idx2   = [epochs2.ch_names.index(ch) for ch in motor_channels
                if ch in epochs2.ch_names]
X_motor2     = epoch_data2[:, motor_idx2, :]
X_ds2        = X_motor2[:, :, ::8]
X_norm2      = (X_ds2 - X_ds2.mean(axis=2, keepdims=True)) / \
               (X_ds2.std(axis=2, keepdims=True) + 1e-8)
X_transformer2 = X_norm2.transpose(0, 2, 1)

print(f"Transformer input shape: {X_transformer2.shape}")

## Retrain Transformer with More Data
Retrain EEG Transformer on extended 45 trial dataset using corrected split then augment pipeline. Fresh model instance ensures unbiased initialization. n_augments reduce to 5 since more original trials are available. Loss descends more gradually than Cell 17, reflecting greater signal diversity across 3 recording sessions. Results 44.44% on 9 test trials.

In [ ]:
# proper split before augmentation
X_tr2, X_te2, y_tr2, y_te2 = tts(
    X_transformer2, labels2,
    test_size=0.2, random_state=42, stratify=labels2
)

X_tr_aug2, y_tr_aug2 = augment_eeg(X_tr2, y_tr2, n_augments=5)

print(f"Train: {len(X_tr_aug2)} (augmented), Test: {len(X_te2)} (original)")

X_train_t2 = torch.FloatTensor(X_tr_aug2)
y_train_t2 = torch.LongTensor(y_tr_aug2)
X_test_t2  = torch.FloatTensor(X_te2)
y_test_t2  = torch.LongTensor(y_te2)

model3 = EEGTransformer(
    n_channels=X_transformer2.shape[2],
    seq_len=X_transformer2.shape[1],
    d_model=32, n_heads=4, n_layers=2, dropout=0.3
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model3.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

train_loader2 = DataLoader(
    TensorDataset(X_train_t2, y_train_t2),
    batch_size=16, shuffle=True
)

n_epochs = 60
train_losses2 = []
print("Training...")
print("─" * 40)

for epoch in range(n_epochs):
    model3.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader2:
        optimizer.zero_grad()
        loss = criterion(model3(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader2)
    train_losses2.append(avg_loss)
    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f}")

model3.eval()
with torch.no_grad():
    _, predicted3 = torch.max(model3(X_test_t2), 1)
    accuracy3 = (predicted3 == y_test_t2).float().mean().item()

print(f"\nTransformer accuracy (90 trials, honest): {accuracy3:.2%}")
print(f"Tested on {len(X_te2)} original unaugmented trials")

plt.figure(figsize=(8, 4))
plt.plot(train_losses2, color='#1D9E75')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Training Loss (Extended Dataset)')
plt.tight_layout()
plt.savefig('../data/transformer_loss_extended.png', dpi=150, bbox_inches='tight')
plt.show()

## Update DB with Results
update the DB with final Transformer result from the extended dataset evaluation. Replaces the Cell 18 estimate with the more reliable 9 trial result. DB is now complete with all six tables populated, ready for Streamlit dashboard to query.

In [ ]:
# --- CELL 21: Update database with final results ---

conn   = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

cursor.execute("DELETE FROM ml_results WHERE model = 'EEGTransformer'")
cursor.execute('''
    INSERT INTO ml_results (model, accuracy, dataset, notes)
    VALUES (?, ?, ?, ?)
''', ('EEGTransformer', round(float(accuracy3), 4),
      'PhysioNet EEG',
      f'2-layer transformer, 3 runs (~90 trials), honest split'))

conn.commit()
cursor.execute('SELECT model, accuracy, notes FROM ml_results ORDER BY accuracy DESC')
print("Final ML results:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]:.2%} — {row[2]}")

conn.close()